# Notebook 1 — Collecte des données et pipeline cloud

**Projet :** Plan your trip with Kayak — Bloc 1 Jedha CDSD
**Auteur :** Aymeric Lahonde

## Objectif

On veut recommander les meilleures destinations françaises pour partir en vacances. Pour ça, on construit un pipeline data complet :

1. Récupérer les coordonnées GPS des 35 villes (Nominatim)
2. Récupérer la météo sur 5 jours (OpenWeatherMap)
3. Calculer un score météo et classer les villes
4. Scraper les hôtels les mieux notés (Booking.com)
5. Stocker les CSV bruts dans un Data Lake S3 (AWS)
6. Charger les données nettoyées dans un Data Warehouse PostgreSQL (RDS)

Le notebook 2 prend ensuite le relais pour les cartes Plotly.


---
## Setup

Toutes les clés sensibles (OpenWeatherMap, AWS, RDS) sont dans `.env` et ne sont jamais commitées sur GitHub.


In [ ]:
# Si certains packages manquent, à exécuter une fois :
# !pip install -r ../requirements.txt


In [ ]:
import os
import re
import io
import time
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

# Pour le scraping Booking
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Pour AWS (S3 + RDS PostgreSQL)
import boto3
from sqlalchemy import create_engine, text

print("Imports OK")


In [ ]:
# Chargement des variables d'environnement (.env du dossier projet)
load_dotenv("../.env")

OWM_API_KEY = os.getenv("OWM_API_KEY")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "eu-west-3")
S3_BUCKET = os.getenv("S3_BUCKET")
RDS_URI = os.getenv("RDS_URI")

# Petit check rapide pour voir si les clés sont bien remplies (pas les placeholders du template)
def is_filled(value, *bad):
    if not value:
        return False
    return not any(p in value for p in bad)

print("OWM API Key :", "OK" if is_filled(OWM_API_KEY, "your_") else "manquante")
print("AWS keys    :", "OK" if is_filled(AWS_ACCESS_KEY_ID, "your_") else "manquantes")
print("S3 Bucket   :", "OK" if is_filled(S3_BUCKET, "your_") else "manquant")
print("RDS URI     :", "OK" if is_filled(RDS_URI, "HOST", "USER", "DBNAME", "PASSWORD") else "manquante")


In [ ]:
# La liste des 35 villes données par Kayak dans l'énoncé
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

print(len(CITIES), "villes à traiter")


---
## 1. Géocodage des 35 villes (Nominatim)

Nominatim est le géocodeur d'OpenStreetMap. Il convertit un nom de ville en coordonnées GPS. C'est gratuit et sans clé API, mais il faut respecter une pause d'1 seconde entre les requêtes.

On a besoin des coordonnées GPS pour ensuite appeler l'API météo (qui demande lat/lon, pas un nom de ville).


In [ ]:
# Fonction qui appelle Nominatim pour une ville
def get_coordinates(city):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": f"{city}, France",  # On précise France pour éviter les ambiguïtés
        "format": "json",
        "limit": 1,
    }
    # Nominatim demande un User-Agent identifiant l'application
    headers = {"User-Agent": "KayakProjectJedha/1.0"}
    
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        if data:
            return {
                "city": city,
                "lat": float(data[0]["lat"]),
                "lon": float(data[0]["lon"]),
            }
    except Exception as e:
        print(f"Erreur GPS pour {city} : {e}")
    return None


In [ ]:
# Test rapide sur une seule ville pour voir le format
get_coordinates("Paris")


In [ ]:
# Boucle sur les 35 villes (avec pause de 1s pour Nominatim)
coords_list = []
for city in tqdm(CITIES, desc="Géolocalisation"):
    result = get_coordinates(city)
    if result:
        coords_list.append(result)
    time.sleep(1)

df_coords = pd.DataFrame(coords_list)
df_coords["city_id"] = range(1, len(df_coords) + 1)

print(len(df_coords), "villes géolocalisées")
df_coords.head()


In [ ]:
# Sauvegarde dans un CSV — on s'en sert plus tard et pour la viz
df_coords.to_csv("../data/cities_coords.csv", index=False)
df_coords.info()


---
## 2. Météo sur 5 jours (OpenWeatherMap)

On utilise l'API Forecast d'OpenWeatherMap (gratuite avec une clé API). Elle renvoie des prévisions par tranches de 3 heures sur 5 jours, soit 40 points de données par ville.

On agrège ces 40 points par jour pour obtenir : température max, probabilité de pluie, volume de pluie, humidité moyenne, vitesse du vent moyenne.


In [ ]:
def get_weather_forecast(lat, lon, city_id, city_name):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OWM_API_KEY,
        "units": "metric",  # Celsius
        "lang": "fr",
        "cnt": 40,  # 40 créneaux 3h = 5 jours
    }
    
    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if r.status_code != 200:
            print(f"Erreur OWM {city_name} : {data.get('message', r.status_code)}")
            return []
        
        # On agrège les créneaux 3h par date
        daily = {}
        for item in data.get("list", []):
            date = pd.to_datetime(item["dt"], unit="s").strftime("%Y-%m-%d")
            if date not in daily:
                daily[date] = {"temps": [], "humidity": [], "pop": [], "rain_mm": 0, "wind_speeds": []}
            daily[date]["temps"].append(item["main"]["temp"])
            daily[date]["humidity"].append(item["main"]["humidity"])
            daily[date]["pop"].append(item.get("pop", 0))
            daily[date]["rain_mm"] += item.get("rain", {}).get("3h", 0)
            daily[date]["wind_speeds"].append(item["wind"]["speed"])
        
        # On construit un enregistrement par jour
        records = []
        for date, vals in daily.items():
            records.append({
                "city_id": city_id,
                "city": city_name,
                "date": date,
                "temp_min": round(min(vals["temps"]), 1),
                "temp_max": round(max(vals["temps"]), 1),
                "humidity": round(sum(vals["humidity"]) / len(vals["humidity"]), 1),
                "pop": round(sum(vals["pop"]) / len(vals["pop"]), 2),
                "rain_mm": round(vals["rain_mm"], 1),
                "wind_speed": round(sum(vals["wind_speeds"]) / len(vals["wind_speeds"]), 1),
            })
        return records
    
    except Exception as e:
        print(f"Erreur météo {city_name} : {e}")
        return []


In [ ]:
# Collecte sur les 35 villes
all_weather = []
for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc="Météo"):
    records = get_weather_forecast(row["lat"], row["lon"], row["city_id"], row["city"])
    all_weather.extend(records)
    time.sleep(0.3)  # Petite pause pour éviter le rate-limit OWM

df_weather = pd.DataFrame(all_weather)
print(len(df_weather), "enregistrements (5 jours x 35 villes)")
df_weather.head()


In [ ]:
# Vérif : on doit avoir 5 jours pour chaque ville
df_weather.groupby("city")["date"].count().describe()


In [ ]:
# Sauvegarde du détail journalier
df_weather.to_csv("../data/weather_daily_detail.csv", index=False)
df_weather.info()


---
## 3. Score météo et classement des villes

On veut un seul score par ville pour pouvoir les classer. On combine 4 critères :

| Critère | Poids | Pourquoi |
|---|---|---|
| Température max moyenne | 40 % | Critère principal du confort |
| Probabilité de pluie | 30 % | Pas envie de pluie en vacances |
| Volume total de pluie | 20 % | Différencier petite pluie / orage |
| Humidité moyenne | 10 % | Affine le confort |

Avant de combiner, on normalise chaque variable entre 0 et 100 (min-max scaling) pour qu'elles soient comparables. Pour les critères où plus petit = mieux (pluie, humidité), on inverse (`100 - norm`).


In [ ]:
# Agrégation par ville sur les 5 jours
df_agg = df_weather.groupby(["city_id", "city"]).agg(
    temp_max_mean=("temp_max", "mean"),
    pop_mean=("pop", "mean"),
    rain_total=("rain_mm", "sum"),
    humidity_mean=("humidity", "mean"),
    wind_mean=("wind_speed", "mean"),
).reset_index()

df_agg.head()


In [ ]:
# Fonction de normalisation min-max
def normalize(series, reverse=False):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    norm = (series - mn) / (mx - mn) * 100
    return (100 - norm) if reverse else norm

# Calcul des 4 scores partiels
df_agg["score_temp"] = normalize(df_agg["temp_max_mean"])
df_agg["score_rain"] = normalize(df_agg["pop_mean"], reverse=True)
df_agg["score_vol"] = normalize(df_agg["rain_total"], reverse=True)
df_agg["score_hum"] = normalize(df_agg["humidity_mean"], reverse=True)

# Score composite pondéré
df_agg["weather_score"] = (
    df_agg["score_temp"] * 0.40
    + df_agg["score_rain"] * 0.30
    + df_agg["score_vol"] * 0.20
    + df_agg["score_hum"] * 0.10
).round(2)

# Classement final
df_ranking = df_agg.sort_values("weather_score", ascending=False).reset_index(drop=True)
df_ranking["rank"] = df_ranking.index + 1

# Fusion avec les coordonnées GPS pour avoir lat/lon dans la table finale
df_weather_final = df_ranking.merge(df_coords[["city_id", "lat", "lon"]], on="city_id")

# Sauvegarde
df_weather_final.to_csv("../data/weather_cities.csv", index=False)

print("Top 10 destinations :")
df_weather_final[["rank", "city", "temp_max_mean", "pop_mean", "rain_total", "weather_score"]].head(10)


---
## 4. Scraping des hôtels Booking.com

Booking.com est un site dynamique : son contenu apparaît après l'exécution de JavaScript. Un simple `requests.get()` ne marche pas. Il faut utiliser **Selenium** pour piloter un vrai navigateur Chrome (en mode headless = sans fenêtre), puis parser le HTML rendu avec BeautifulSoup.

Pour chaque ville, on récupère les 20 premiers hôtels avec :
- nom de l'hôtel
- URL de la fiche
- score utilisateurs
- description / adresse

Note : Booking change ses sélecteurs CSS régulièrement, donc le code peut casser et nécessite des ajustements (c'est la difficulté du scraping).


In [ ]:
# Crée un driver Chrome headless
def create_driver():
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    # On simule un vrai navigateur (sinon Booking détecte le bot)
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts,
    )


In [ ]:
# Scrape les hôtels d'une ville
def scrape_city_hotels(driver, city_name, city_id, city_lat, city_lon, n_hotels=20):
    hotels = []
    city_enc = city_name.replace(" ", "+")
    url = f"https://www.booking.com/searchresults.html?ss={city_enc}&lang=fr&dest_type=city"
    
    driver.get(url)
    time.sleep(4)  # On attend le rendu JavaScript
    
    # Fermeture du popup cookies si présent
    try:
        btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        btn.click()
        time.sleep(1)
    except Exception:
        pass
    
    # Attendre que les cards des hôtels apparaissent
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="property-card"]'))
        )
    except Exception:
        pass
    
    # Parsing avec BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.find_all("div", {"data-testid": "property-card"})
    
    for card in cards[:n_hotels]:
        # Nom + URL
        link_tag = card.find("a", {"data-testid": "title-link"})
        name = link_tag.get_text(strip=True) if link_tag else None
        if not name:
            continue
        
        href = link_tag.get("href", "") if link_tag else ""
        if href and not href.startswith("http"):
            href = "https://www.booking.com" + href
        hotel_url = href.split("?")[0] if ".html" in href else href
        
        # Score utilisateur (extrait depuis le texte type "Avec une note de 9,2")
        score = None
        score_tag = card.find("div", {"data-testid": "review-score"})
        if score_tag:
            m = re.search(r"(\d+)[,.](\d+)", score_tag.get_text())
            if m:
                score = float(f"{m.group(1)}.{m.group(2)}")
        
        # Adresse / description
        addr_tag = card.find(["span", "div"], {"data-testid": "address"})
        description = addr_tag.get_text(strip=True) if addr_tag else ""
        
        hotels.append({
            "city_id": city_id,
            "city": city_name,
            "hotel_name": name,
            "url": hotel_url,
            "lat": city_lat,  # Coords du centre-ville (on les disperse plus tard)
            "lon": city_lon,
            "score": score,
            "description": description,
        })
    
    return hotels


In [ ]:
# Lancement du scraping sur les 35 villes (~9-10 min)
driver = create_driver()
all_hotels = []
np.random.seed(42)  # Pour la reproductibilité de la dispersion GPS

for _, row in tqdm(df_coords.iterrows(), total=len(df_coords), desc="Hôtels Booking"):
    hotels = scrape_city_hotels(
        driver, row["city"], row["city_id"], row["lat"], row["lon"], n_hotels=20
    )
    all_hotels.extend(hotels)
    time.sleep(2)  # Pause pour éviter le blocage IP

driver.quit()

df_hotels = pd.DataFrame(all_hotels)
df_hotels["score"] = pd.to_numeric(df_hotels["score"], errors="coerce")

# Tous les hôtels d'une même ville ont la même lat/lon (centre-ville),
# on disperse légèrement pour que la carte n'ait pas tous les points superposés
n = len(df_hotels)
df_hotels["lat"] += np.random.uniform(-0.02, 0.02, n)
df_hotels["lon"] += np.random.uniform(-0.02, 0.02, n)
df_hotels["hotel_id"] = range(1, n + 1)

df_hotels.to_csv("../data/hotels_booking.csv", index=False)

print(len(df_hotels), "hôtels collectés")
print("Avec score :", df_hotels["score"].notna().sum())
df_hotels.head()


In [ ]:
# Top 10 hôtels par score Booking
(
    df_hotels.dropna(subset=["score"])
    .sort_values("score", ascending=False)
    [["hotel_name", "city", "score"]]
    .head(10)
)


---
## 5. Data Lake S3 (AWS)

Maintenant que les CSV sont produits en local, on les uploade sur un bucket S3. C'est notre "Data Lake" : un stockage cloud pour les données brutes.

Convention : les fichiers bruts vont dans le préfixe `raw/` du bucket. Plus tard on pourrait avoir `processed/` pour les données nettoyées et `warehouse_ready/` pour les données prêtes à charger en RDS.


In [ ]:
# Connexion au client S3 avec boto3
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

# Upload des 4 CSV sous le préfixe raw/
files_to_upload = [
    "cities_coords.csv",
    "weather_cities.csv",
    "weather_daily_detail.csv",
    "hotels_booking.csv",
]

for fname in files_to_upload:
    src = f"../data/{fname}"
    if os.path.exists(src):
        s3_client.upload_file(Filename=src, Bucket=S3_BUCKET, Key=f"raw/{fname}")
        print(f"Upload : s3://{S3_BUCKET}/raw/{fname}")


In [ ]:
# Vérif : on liste le contenu du bucket
print(f"Contenu du bucket {S3_BUCKET} :")
for obj in s3_client.list_objects_v2(Bucket=S3_BUCKET).get("Contents", []):
    size_kb = obj["Size"] / 1024
    print(f"  {obj['Key']:<40} {size_kb:>8.1f} KB")


---
## 6. ETL vers RDS PostgreSQL (Data Warehouse)

Le Data Lake stocke les données brutes. Le Data Warehouse stocke les données nettoyées et structurées en SQL. C'est ce que les équipes data analysts vont utiliser pour leurs requêtes.

**ETL** :
- **E**xtract — on lit les CSV depuis S3
- **T**ransform — on sélectionne les colonnes utiles, on type les données
- **L**oad — on insère dans 2 tables PostgreSQL : `cities_weather` et `hotels`

**Instance RDS utilisée :** `kayak-jedha` (db.t3.micro, PostgreSQL 16, eu-west-3, free tier).


In [ ]:
# EXTRACT : on relit les CSV depuis S3 (pour valider le pipeline cloud-only)
def read_csv_from_s3(s3_key):
    obj = s3_client.get_object(Bucket=S3_BUCKET, Key=s3_key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))
    print(f"  {s3_key} : {df.shape[0]} lignes")
    return df

print("Extract depuis S3 :")
df_weather_s3 = read_csv_from_s3("raw/weather_cities.csv")
df_hotels_s3 = read_csv_from_s3("raw/hotels_booking.csv")


In [ ]:
# TRANSFORM : sélection des colonnes utiles + arrondis pour la lisibilité
df_weather_clean = df_weather_s3[[
    "city_id", "city", "lat", "lon",
    "temp_max_mean", "pop_mean", "rain_total",
    "humidity_mean", "weather_score", "rank",
]].copy()
df_weather_clean["temp_max_mean"] = df_weather_clean["temp_max_mean"].round(1)
df_weather_clean["weather_score"] = df_weather_clean["weather_score"].round(2)

df_hotels_clean = df_hotels_s3[[
    "city_id", "city", "hotel_name", "url",
    "lat", "lon", "score", "description",
]].copy()
df_hotels_clean["score"] = pd.to_numeric(df_hotels_clean["score"], errors="coerce")
df_hotels_clean["hotel_id"] = range(1, len(df_hotels_clean) + 1)

print("Tables nettoyées :")
print("  cities_weather :", df_weather_clean.shape)
print("  hotels         :", df_hotels_clean.shape)

df_weather_clean.sort_values("rank").head()


In [ ]:
# LOAD : insertion dans PostgreSQL via SQLAlchemy
engine = create_engine(RDS_URI)

# On supprime les tables existantes pour repartir propre
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS hotels"))
    conn.execute(text("DROP TABLE IF EXISTS cities_weather"))
    conn.commit()

# Insertion des données
df_weather_clean.to_sql("cities_weather", engine, if_exists="replace", index=False)
df_hotels_clean.to_sql("hotels", engine, if_exists="replace", index=False)

print("Tables créées :")
print(f"  cities_weather : {len(df_weather_clean)} lignes")
print(f"  hotels         : {len(df_hotels_clean)} lignes")


In [ ]:
# Vérification : on requête directement RDS pour confirmer
with engine.connect() as conn:
    top5 = pd.read_sql(
        text(
            "SELECT rank, city, temp_max_mean, pop_mean, rain_total, weather_score "
            "FROM cities_weather ORDER BY rank LIMIT 5"
        ),
        conn,
    )
    print("Top 5 destinations (depuis RDS) :")
    display(top5)
    
    top_hotels = pd.read_sql(
        text(
            "SELECT hotel_name, city, score FROM hotels "
            "WHERE score IS NOT NULL ORDER BY score DESC LIMIT 5"
        ),
        conn,
    )
    print("\nTop 5 hôtels (depuis RDS) :")
    display(top_hotels)


---
## Bilan

On a fait tourner le pipeline complet :

| Étape | Sortie |
|---|---|
| Géocodage | `cities_coords.csv` (35 villes) |
| Météo détail | `weather_daily_detail.csv` (175 lignes : 5j × 35 villes) |
| Score météo | `weather_cities.csv` (35 villes classées) |
| Scraping | `hotels_booking.csv` (~700 hôtels) |
| Data Lake | bucket S3 `raw/` avec les 4 CSV |
| Data Warehouse | RDS PostgreSQL avec 2 tables (`cities_weather` + `hotels`) |

**Top 5 destinations (météo février 2026)** : Aix-en-Provence, Avignon, Marseille, Paris, Grenoble.

Le notebook 2 prend ensuite le relais pour produire les 2 cartes interactives (Top 5 destinations + Top 20 hôtels).

**Pistes d'amélioration :**
- Pondérer le score par la saison (un Marseille pluvieux en janvier vs un Paris ensoleillé en juillet ne se valent pas)
- Croiser avec le prix moyen des hôtels (autre champ à scraper)
- Automatiser le pipeline (Airflow / cron) pour rafraîchir les données quotidiennement
- Restreindre le security group RDS à une IP spécifique au lieu de `0.0.0.0/0`
